# FreeSurfer 8.1.0 Infant Processing Pipeline - Step-by-Step Visualization

This notebook demonstrates the **FreeSurfer 8.1.0 infant processing pipeline** with visualizations at each step.

**Key Difference from FS7**: This workflow is **dramatically simpler** - just 2 commands instead of 10+ steps!

## Table of Contents
1. Setup and Configuration
2. Input Data Inspection
3. FreeSurfer 8.1.0 Processing (Integrated Infant Pipeline)
4. Visualization of Processing Stages
5. Label Convention Check (Automatic in FS8)
6. Quality Control
7. Comparison with FS7 Workflow

---

## Why FreeSurfer 8.1.0?

### FS7 + Infant FreeSurfer Workflow:
```bash
# 10+ steps:
recon-all -all          # 8-12 hours
prepare for iFS
infant_recon_all        # 2-4 hours  
remap labels (9→10)     # manual!
remap labels (48→49)    # manual!
generate wm.mgz         # manual!
autorecon2-end          # 4-6 hours
autorecon3              # 4-6 hours
```

### FS8 Workflow:
```bash
# 2 steps:
recon-all -i T1w.nii.gz -subjid sub-01
infant_recon_all -s sub-01 -age 6 -all    # 20-30 hours, done!
```

**Everything is automatic in FS8!**

## 1. Setup and Configuration

In [ ]:
# Import libraries
import os
import json
import subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from nilearn import plotting, image
from IPython.display import display, HTML
import pandas as pd
from pathlib import Path

# Set plotting parameters
plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['font.size'] = 10

print("✓ Libraries imported successfully")

In [ ]:
# Define paths - CONFIGURED FOR YOUR DATA
INPUT_T1W = "/data02/share/bin-wu/data/human/brain/harvard_mri/raw/new_england/ds006169-1.0.3/sub-01/ses-03/anat/sub-01_ses-03_T1w.nii.gz"
INPUT_JSON = "/data02/share/bin-wu/data/human/brain/harvard_mri/raw/new_england/ds006169-1.0.3/sub-01/ses-03/anat/sub-01_ses-03_T1w.json"

# FreeSurfer 8.1.0 directory (NOTE: Different from FS7!)
SUBJECTS_DIR = "/data02/share/bin-wu/data/human/brain/harvard_mri/processed/freesurfer_fs8"
SUBJECT_ID = "sub-01_ses-03"

# Age in months (adjust for your subject)
AGE_MONTHS = 6

# Create directories if needed
os.makedirs(SUBJECTS_DIR, exist_ok=True)

# Verify input files exist
print(f"Input T1w exists: {os.path.exists(INPUT_T1W)}")
print(f"Input JSON exists: {os.path.exists(INPUT_JSON)}")
print(f"\nSubjects directory: {SUBJECTS_DIR}")
print(f"Subject ID: {SUBJECT_ID}")
print(f"Age: {AGE_MONTHS} months")

In [ ]:
# Set FreeSurfer 8.1.0 environment
# NOTE: You need to adjust this path for your system

os.environ['FREESURFER_HOME'] = '/usr/local/freesurfer-8.1.0'  # Adjust this!
os.environ['SUBJECTS_DIR'] = SUBJECTS_DIR

# Verify FreeSurfer 8 is available
try:
    result = subprocess.run(['which', 'infant_recon_all'], capture_output=True, text=True)
    if result.stdout:
        print(f"✓ infant_recon_all found: {result.stdout.strip()}")
    else:
        print("⚠️ infant_recon_all not found in PATH")
        print("Please ensure FreeSurfer 8.1.0 is installed and sourced:")
        print("  source $FREESURFER_HOME/SetUpFreeSurfer.sh")
except:
    print("⚠️ Could not check for infant_recon_all")

## 2. Input Data Inspection

In [ ]:
# Load and display JSON metadata
with open(INPUT_JSON, 'r') as f:
    metadata = json.load(f)

print("=" * 60)
print("MRI ACQUISITION PARAMETERS")
print("=" * 60)
for key, value in sorted(metadata.items()):
    print(f"{key:30s}: {value}")

In [ ]:
# Load and inspect T1w image
t1w_img = nib.load(INPUT_T1W)
t1w_data = t1w_img.get_fdata()

print("=" * 60)
print("T1W IMAGE PROPERTIES")
print("=" * 60)
print(f"Shape:        {t1w_data.shape}")
print(f"Voxel size:   {t1w_img.header.get_zooms()[:3]} mm")
print(f"Data type:    {t1w_data.dtype}")
print(f"Value range:  {t1w_data.min():.1f} - {t1w_data.max():.1f}")

In [ ]:
# Visualize input T1w
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

mid_sag = t1w_data.shape[0] // 2
mid_cor = t1w_data.shape[1] // 2
mid_axi = t1w_data.shape[2] // 2

axes[0].imshow(t1w_data[mid_sag, :, :].T, cmap='gray', origin='lower')
axes[0].set_title('Sagittal View', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(t1w_data[:, mid_cor, :].T, cmap='gray', origin='lower')
axes[1].set_title('Coronal View', fontsize=14, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(t1w_data[:, :, mid_axi].T, cmap='gray', origin='lower')
axes[2].set_title('Axial View', fontsize=14, fontweight='bold')
axes[2].axis('off')

plt.suptitle('INPUT T1w IMAGE', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Input image looks good!")

## 3. FreeSurfer 8.1.0 Processing Commands

This section shows the **actual commands** to run. The processing takes ~20-30 hours, so we'll visualize the outputs after processing completes.

In [ ]:
# Display the commands (DO NOT RUN in notebook - takes 20-30 hours!)

print("=" * 80)
print("FREESURFER 8.1.0 PROCESSING COMMANDS")
print("=" * 80)
print("\nRun these commands in your terminal (NOT in this notebook!):\n")
print("# Step 1: Import T1w (import-only, no auto-processing)")
print(f"recon-all -i {INPUT_T1W} \\")
print(f"          -subjid {SUBJECT_ID} \\")
print(f"          -sd {SUBJECTS_DIR} \\")
print(f"          -noskullstrip")
print()
print("# Step 2: Create orig.mgz (required for infant_recon_all)")
print(f"mri_convert {SUBJECTS_DIR}/{SUBJECT_ID}/mri/orig/001.mgz \\")
print(f"            {SUBJECTS_DIR}/{SUBJECT_ID}/mri/orig.mgz")
print()
print("# Step 3: Run FreeSurfer 8.1.0 infant processing (20-30 hours)")
print(f"infant_recon_all -s {SUBJECT_ID} \\")
print(f"                 -age {AGE_MONTHS} \\")
print(f"                 -all")
print()
print("That's it! Just 3 simple commands.")
print("\n" + "=" * 80)
print("\nNOTE: The -noskullstrip flag prevents recon-all from auto-running")
print("      synthstrip (which would fail). infant_recon_all handles all processing.")
print("=" * 80)

## 4. Visualization of Processing Stages

After processing completes, we can visualize the outputs.

**Note**: Run this section AFTER the processing has finished.

In [ ]:
# Check if processing has been run
subject_path = Path(SUBJECTS_DIR) / SUBJECT_ID

if not subject_path.exists():
    print(f"⚠️ Subject directory not found: {subject_path}")
    print("\nPlease run the processing commands first (see cell above)")
else:
    print(f"✓ Subject directory found: {subject_path}")
    print("\nChecking for key files...")
    
    files_to_check = {
        'orig.mgz': subject_path / 'mri' / 'orig.mgz',
        'T1.mgz': subject_path / 'mri' / 'T1.mgz',
        'brainmask.mgz': subject_path / 'mri' / 'brainmask.mgz',
        'aseg.mgz': subject_path / 'mri' / 'aseg.mgz',
        'wm.mgz': subject_path / 'mri' / 'wm.mgz',
    }
    
    for name, path in files_to_check.items():
        exists = path.exists()
        status = "✓" if exists else "✗"
        print(f"  {status} {name}")
    
    # Check if processing is complete
    if all(path.exists() for path in files_to_check.values()):
        print("\n✓ Processing appears complete! Proceeding with visualization...")
    else:
        print("\n⚠️ Some files are missing - processing may be incomplete")

### 4.1 Original Image (orig.mgz)

In [ ]:
orig_file = subject_path / 'mri' / 'orig.mgz'

if orig_file.exists():
    orig_img = nib.load(orig_file)
    orig_data = orig_img.get_fdata()
    
    print(f"orig.mgz shape: {orig_data.shape}")
    print(f"orig.mgz voxel size: {orig_img.header.get_zooms()[:3]} mm")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    mid_sag = orig_data.shape[0] // 2
    mid_cor = orig_data.shape[1] // 2
    mid_axi = orig_data.shape[2] // 2
    
    axes[0].imshow(orig_data[mid_sag, :, :].T, cmap='gray', origin='lower')
    axes[0].set_title('Sagittal')
    axes[0].axis('off')
    
    axes[1].imshow(orig_data[:, mid_cor, :].T, cmap='gray', origin='lower')
    axes[1].set_title('Coronal')
    axes[1].axis('off')
    
    axes[2].imshow(orig_data[:, :, mid_axi].T, cmap='gray', origin='lower')
    axes[2].set_title('Axial')
    axes[2].axis('off')
    
    plt.suptitle('ORIGINAL IMAGE (orig.mgz)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ orig.mgz not found")

### 4.2 Normalized T1 (T1.mgz)

In [ ]:
t1_file = subject_path / 'mri' / 'T1.mgz'

if t1_file.exists():
    t1_img = nib.load(t1_file)
    t1_data = t1_img.get_fdata()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    mid_sag = t1_data.shape[0] // 2
    mid_cor = t1_data.shape[1] // 2
    mid_axi = t1_data.shape[2] // 2
    
    axes[0].imshow(t1_data[mid_sag, :, :].T, cmap='gray', origin='lower')
    axes[0].set_title('Sagittal')
    axes[0].axis('off')
    
    axes[1].imshow(t1_data[:, mid_cor, :].T, cmap='gray', origin='lower')
    axes[1].set_title('Coronal')
    axes[1].axis('off')
    
    axes[2].imshow(t1_data[:, :, mid_axi].T, cmap='gray', origin='lower')
    axes[2].set_title('Axial')
    axes[2].axis('off')
    
    plt.suptitle('NORMALIZED T1 (T1.mgz)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ T1.mgz not found")

### 4.3 Brain Mask (brainmask.mgz)

In [ ]:
brainmask_file = subject_path / 'mri' / 'brainmask.mgz'

if brainmask_file.exists() and t1_file.exists():
    brainmask_img = nib.load(brainmask_file)
    brainmask_data = brainmask_img.get_fdata()
    
    # Calculate brain volume
    brain_voxels = np.sum(brainmask_data > 0)
    voxel_vol = np.prod(brainmask_img.header.get_zooms()[:3])
    brain_volume_ml = brain_voxels * voxel_vol / 1000
    
    print(f"Brain volume: {brain_volume_ml:.1f} ml")
    print(f"Expected range for {AGE_MONTHS}-month infant: ~400-1000 ml")
    
    # Overlay on T1
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    mid_sag = t1_data.shape[0] // 2
    mid_cor = t1_data.shape[1] // 2
    mid_axi = t1_data.shape[2] // 2
    
    # Show T1 with brain mask overlay
    axes[0].imshow(t1_data[mid_sag, :, :].T, cmap='gray', origin='lower')
    axes[0].imshow(brainmask_data[mid_sag, :, :].T, cmap='hot', alpha=0.3, origin='lower')
    axes[0].set_title('Sagittal')
    axes[0].axis('off')
    
    axes[1].imshow(t1_data[:, mid_cor, :].T, cmap='gray', origin='lower')
    axes[1].imshow(brainmask_data[:, mid_cor, :].T, cmap='hot', alpha=0.3, origin='lower')
    axes[1].set_title('Coronal')
    axes[1].axis('off')
    
    axes[2].imshow(t1_data[:, :, mid_axi].T, cmap='gray', origin='lower')
    axes[2].imshow(brainmask_data[:, :, mid_axi].T, cmap='hot', alpha=0.3, origin='lower')
    axes[2].set_title('Axial')
    axes[2].axis('off')
    
    plt.suptitle('BRAIN MASK OVERLAY (red = brain)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ brainmask.mgz or T1.mgz not found")

### 4.4 Segmentation (aseg.mgz)

**IMPORTANT**: FreeSurfer 8 uses **standard labels** automatically!
- Left thalamus: **10** (not 9)
- Right thalamus: **49** (not 48)

No manual remapping needed!

In [ ]:
aseg_file = subject_path / 'mri' / 'aseg.mgz'

if aseg_file.exists():
    aseg_img = nib.load(aseg_file)
    aseg_data = aseg_img.get_fdata()
    
    # Check labels
    unique_labels = np.unique(aseg_data[aseg_data > 0])
    print(f"Number of labels: {len(unique_labels)}")
    print(f"Label range: {aseg_data.min():.0f} - {aseg_data.max():.0f}")
    
    # Check for key structures
    print("\nKey structures:")
    key_labels = {
        2: 'Left Cerebral White Matter',
        41: 'Right Cerebral White Matter',
        3: 'Left Cerebral Cortex',
        42: 'Right Cerebral Cortex',
        10: 'Left Thalamus (FS8 standard)',  # Note: 10 not 9!
        49: 'Right Thalamus (FS8 standard)', # Note: 49 not 48!
    }
    
    for label, name in key_labels.items():
        count = np.sum(aseg_data == label)
        status = "✓" if count > 0 else "✗"
        print(f"  {status} Label {label:2d}: {name:35s} ({count:6d} voxels)")
    
    # Check for old iFS labels (should be absent in FS8!)
    print("\nChecking for old iFS labels (should be 0 in FS8):")
    old_labels = {9: 'Old iFS left thalamus', 48: 'Old iFS right thalamus'}
    for label, name in old_labels.items():
        count = np.sum(aseg_data == label)
        if count > 0:
            print(f"  ✗ Label {label}: {count} voxels (UNEXPECTED IN FS8!)")
        else:
            print(f"  ✓ Label {label}: 0 voxels (correct - FS8 uses standard labels)")
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    mid_sag = aseg_data.shape[0] // 2
    mid_cor = aseg_data.shape[1] // 2
    mid_axi = aseg_data.shape[2] // 2
    
    axes[0].imshow(aseg_data[mid_sag, :, :].T, cmap='tab20', origin='lower')
    axes[0].set_title('Sagittal')
    axes[0].axis('off')
    
    axes[1].imshow(aseg_data[:, mid_cor, :].T, cmap='tab20', origin='lower')
    axes[1].set_title('Coronal')
    axes[1].axis('off')
    
    axes[2].imshow(aseg_data[:, :, mid_axi].T, cmap='tab20', origin='lower')
    axes[2].set_title('Axial')
    axes[2].axis('off')
    
    plt.suptitle('SEGMENTATION (aseg.mgz) - FS8 Standard Labels', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ aseg.mgz not found")

## 5. Label Convention Check

This demonstrates a **key advantage** of FreeSurfer 8:

**FS7 + iFS**: Requires manual remapping (9→10, 48→49)  
**FS8**: Uses standard labels automatically ✓

In [ ]:
if aseg_file.exists():
    print("=" * 70)
    print("THALAMUS LABEL CHECK - FS8 AUTOMATIC HANDLING")
    print("=" * 70)
    
    # Check thalamus labels
    left_thal_old = np.sum(aseg_data == 9)   # iFS convention (should be 0)
    left_thal_new = np.sum(aseg_data == 10)  # FS standard (should be >0)
    right_thal_old = np.sum(aseg_data == 48) # iFS convention (should be 0)
    right_thal_new = np.sum(aseg_data == 49) # FS standard (should be >0)
    
    print("\nLeft Thalamus:")
    print(f"  Label  9 (iFS):  {left_thal_old:6d} voxels {'(SHOULD BE 0)' if left_thal_old > 0 else '✓'}")
    print(f"  Label 10 (FS):   {left_thal_new:6d} voxels {'(SHOULD BE >0)' if left_thal_new == 0 else '✓'}")
    
    print("\nRight Thalamus:")
    print(f"  Label 48 (iFS):  {right_thal_old:6d} voxels {'(SHOULD BE 0)' if right_thal_old > 0 else '✓'}")
    print(f"  Label 49 (FS):   {right_thal_new:6d} voxels {'(SHOULD BE >0)' if right_thal_new == 0 else '✓'}")
    
    print("\n" + "=" * 70)
    if left_thal_old == 0 and right_thal_old == 0 and left_thal_new > 0 and right_thal_new > 0:
        print("✓✓✓ PERFECT! FreeSurfer 8 used standard labels automatically!")
        print("    No manual remapping needed!")
    else:
        print("⚠️ Unexpected label values - check processing")
    print("=" * 70)
    
    # Visualize comparison
    print("\n\nFS7 + iFS vs FS8 Label Handling:\n")
    print("┌─────────────────────────────────────────────────────────────┐")
    print("│ FS7 + Infant FreeSurfer (OLD WAY):                          │")
    print("│   1. iFS creates aseg with labels 9, 48                     │")
    print("│   2. Manual remapping required:                             │")
    print("│      mri_binarize --match 9 --replace 10                    │")
    print("│      mri_binarize --match 48 --replace 49                   │")
    print("│   3. Easy to make mistakes!                                 │")
    print("├─────────────────────────────────────────────────────────────┤")
    print("│ FS8 (NEW WAY):                                              │")
    print("│   1. FS8 creates aseg with labels 10, 49 automatically      │")
    print("│   2. No remapping needed!                                   │")
    print("│   3. Automatic and error-free!                              │")
    print("└─────────────────────────────────────────────────────────────┘")

## 6. Quality Control

Run comprehensive QC on the outputs.

In [ ]:
# Run automated QC script
print("Running automated QC...\n")

qc_script = Path("detailed_qc_visualization_fs8.py")
if qc_script.exists():
    cmd = f"python {qc_script} {SUBJECTS_DIR} {SUBJECT_ID}"
    print(f"Command: {cmd}\n")
    print("(Run this in terminal for full output)")
    print("\nThis will generate:")
    print("  - PNG images for all processing stages")
    print("  - qc_report_fs8.json with all checks")
    print("  - Pass/Warning/Error status")
else:
    print(f"⚠️ QC script not found: {qc_script}")
    print("Make sure you're in the FS8 directory")

## 7. Comparison: FS7 vs FS8 Workflow

Visual comparison of the two approaches.

In [ ]:
from IPython.display import HTML

html = """
<style>
.comparison {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 20px;
    margin: 20px 0;
}
.fs7-box {
    background-color: #fff3cd;
    border: 2px solid #ffc107;
    border-radius: 8px;
    padding: 15px;
}
.fs8-box {
    background-color: #d1ecf1;
    border: 2px solid #0c5460;
    border-radius: 8px;
    padding: 15px;
}
.step {
    margin: 5px 0;
    padding: 5px;
    background-color: rgba(255,255,255,0.5);
    border-radius: 4px;
}
h3 { margin-top: 0; }
</style>

<div class="comparison">
    <div class="fs7-box">
        <h3>🔴 FreeSurfer 7.3 + Infant FreeSurfer</h3>
        <p><strong>Complexity: HIGH</strong></p>
        <div class="step">1. recon-all -all (8-12h)</div>
        <div class="step">2. Clean up files</div>
        <div class="step">3. Prepare for iFS</div>
        <div class="step">4. Run infant_recon_all (2-4h)</div>
        <div class="step">5. Copy iFS aseg</div>
        <div class="step">6. Remap label 9 → 10 (manual!)</div>
        <div class="step">7. Remap label 48 → 49 (manual!)</div>
        <div class="step">8. Generate wm.mgz (manual!)</div>
        <div class="step">9. Copy transforms</div>
        <div class="step">10. Run autorecon2-end (4-6h)</div>
        <div class="step">11. Run autorecon3 (4-6h)</div>
        <p><strong>Total: 10+ steps, 2 directories, ~20-30 hours</strong></p>
    </div>
    
    <div class="fs8-box">
        <h3>🟢 FreeSurfer 8.1.0</h3>
        <p><strong>Complexity: LOW</strong></p>
        <div class="step">1. recon-all -i T1w.nii.gz (5 min)</div>
        <div class="step">2. infant_recon_all -age N -all (20-30h)</div>
        <br><br><br><br><br><br><br><br><br>
        <p><strong>Total: 2 steps, 1 directory, ~20-30 hours</strong></p>
        <p style="color: green; font-weight: bold;">✓ Label remapping automatic!</p>
        <p style="color: green; font-weight: bold;">✓ WM generation automatic!</p>
        <p style="color: green; font-weight: bold;">✓ Everything in one place!</p>
    </div>
</div>

<h3 style="margin-top: 30px;">Key Differences:</h3>
<table style="width: 100%; border-collapse: collapse; margin: 10px 0;">
    <tr style="background-color: #f0f0f0;">
        <th style="border: 1px solid #ddd; padding: 8px;">Feature</th>
        <th style="border: 1px solid #ddd; padding: 8px;">FS7 + iFS</th>
        <th style="border: 1px solid #ddd; padding: 8px;">FS8</th>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 8px;">Installation</td>
        <td style="border: 1px solid #ddd; padding: 8px;">2 systems</td>
        <td style="border: 1px solid #ddd; padding: 8px; background-color: #d4edda;">1 system</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 8px;">Steps</td>
        <td style="border: 1px solid #ddd; padding: 8px;">10+ steps</td>
        <td style="border: 1px solid #ddd; padding: 8px; background-color: #d4edda;">2 steps</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 8px;">Label Remapping</td>
        <td style="border: 1px solid #ddd; padding: 8px;">Manual</td>
        <td style="border: 1px solid #ddd; padding: 8px; background-color: #d4edda;">Automatic</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 8px;">Directories</td>
        <td style="border: 1px solid #ddd; padding: 8px;">2 (FS + iFS)</td>
        <td style="border: 1px solid #ddd; padding: 8px; background-color: #d4edda;">1</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 8px;">Error Potential</td>
        <td style="border: 1px solid #ddd; padding: 8px;">Higher (manual steps)</td>
        <td style="border: 1px solid #ddd; padding: 8px; background-color: #d4edda;">Lower (automatic)</td>
    </tr>
</table>
"""

display(HTML(html))

## Summary

This notebook demonstrated the FreeSurfer 8.1.0 infant processing pipeline.

### Key Takeaways:

1. **Much Simpler**: 2 commands vs 10+ steps
2. **Automatic Labels**: No manual remapping needed (9→10, 48→49)
3. **Single Directory**: Everything in one place
4. **Latest Algorithms**: Updated processing methods (2023+)
5. **Same Quality**: Equivalent outputs to FS7 version

### When to Use FS8:

✅ New projects  
✅ Want simplicity  
✅ Learning/teaching  
✅ Don't need exact study replication  

### When to Use FS7:

✅ Exact study replication  
✅ Consistency with existing FS7 data  
✅ Validation requirements  

### Next Steps:

1. Run the processing commands (see Section 3)
2. Wait ~20-30 hours for completion
3. Run QC visualization (Section 6)
4. Extract statistics
5. Analyze results

### Additional Resources:

- **FS8 User Guide**: `README_FS8_INFANT_PIPELINE.md`
- **FS7 vs FS8 Comparison**: `FS7_vs_FS8_COMPARISON.md`
- **QC Script**: `detailed_qc_visualization_fs8.py`
- **FreeSurfer 8 Docs**: https://surfer.nmr.mgh.harvard.edu/fswiki/FreeSurfer8